<a href="https://colab.research.google.com/github/uvindu827/J26-DS-350/blob/Indhi_Component2_CHL_Forecast/31_9P_16T_3D_Spatial_Temporal_CHL_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================================
# STEP 31.9P-16T-17
# 3D SPATIAL-TEMPORAL CHL VISUALIZATION
# ENVIRONMENT & DATA VERIFICATION
# ================================================================

import os
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 31.9P-16T-17")
print("3D SPATIAL-TEMPORAL CHL VISUALIZATION")
print("ENVIRONMENT & DATA VERIFICATION")
print("=" * 70)


# ================================================================
# PROJECT PATHS
# ================================================================

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization",
    "final_tuned_candidate"
)

FINAL_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "xgboost_final_tuned_candidate.json"
)

FINAL_CONFIG_PATH = os.path.join(
    MODEL_DIR,
    "xgboost_final_tuned_candidate_configuration.json"
)

FEATURE_LIST_PATH = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_feature_columns.csv"
)

TEST_DATA_PATH = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_test.csv"
)

TUNED_TEST_PREDICTIONS_PATH = os.path.join(
    MODEL_DIR,
    "final_test_evaluation",
    "xgboost_final_tuned_test_predictions.csv"
)

SHAP_DIR = os.path.join(
    MODEL_DIR,
    "shap_explainability"
)

OUTPUT_DIR = os.path.join(
    MODEL_DIR,
    "3d_spatial_temporal_visualization"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print(f"\nProject:")
print(PROJECT_DIR)

print(f"\nFinal tuned model:")
print(FINAL_MODEL_PATH)

print(f"\nFinal feature list:")
print(FEATURE_LIST_PATH)

print(f"\nFinal test dataset:")
print(TEST_DATA_PATH)

print(f"\nSaved tuned test predictions:")
print(TUNED_TEST_PREDICTIONS_PATH)

print(f"\nSHAP directory:")
print(SHAP_DIR)

print(f"\n3D visualization output directory:")
print(OUTPUT_DIR)


# ================================================================
# ENVIRONMENT CHECK
# ================================================================

print("\n" + "=" * 70)
print("VISUALIZATION ENVIRONMENT CHECK")
print("=" * 70)

try:
    import plotly
    print(f"Plotly version: {plotly.__version__}")
    PLOTLY_AVAILABLE = True
except Exception as e:
    print("Plotly is not available.")
    print(f"Error: {e}")
    PLOTLY_AVAILABLE = False

try:
    import matplotlib
    print(f"Matplotlib version: {matplotlib.__version__}")
    MATPLOTLIB_AVAILABLE = True
except Exception as e:
    print("Matplotlib is not available.")
    print(f"Error: {e}")
    MATPLOTLIB_AVAILABLE = False

try:
    import sklearn
    print(f"Scikit-learn version: {sklearn.__version__}")
    SKLEARN_AVAILABLE = True
except Exception as e:
    print("Scikit-learn is not available.")
    print(f"Error: {e}")
    SKLEARN_AVAILABLE = False


if not PLOTLY_AVAILABLE:
    raise ImportError(
        "Plotly is required for the interactive 3D visualization."
    )

print("\n✓ Visualization environment verified.")


# ================================================================
# REQUIRED FILE CHECK
# ================================================================

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {
    "Final tuned XGBoost": FINAL_MODEL_PATH,
    "Final configuration": FINAL_CONFIG_PATH,
    "Final feature list": FEATURE_LIST_PATH,
    "Final test dataset": TEST_DATA_PATH,
    "Tuned test predictions": TUNED_TEST_PREDICTIONS_PATH,
}

all_files_available = True

for name, path in required_files.items():
    exists = os.path.isfile(path)

    print(f"{name:<30}: {exists}")

    if not exists:
        all_files_available = False
        print(f"  Missing: {path}")

if not all_files_available:
    raise FileNotFoundError(
        "One or more required visualization files are missing."
    )

print("\n✓ All required visualization files found.")


# ================================================================
# LOAD FINAL FEATURE LIST
# ================================================================

print("\n" + "=" * 70)
print("LOADING FINAL FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(FEATURE_LIST_PATH)

print(f"Feature list shape: {feature_df.shape}")

# Identify feature-name column safely
possible_feature_columns = [
    "feature",
    "feature_name",
    "Feature",
    "Feature_Name"
]

feature_column = None

for col in possible_feature_columns:
    if col in feature_df.columns:
        feature_column = col
        break

if feature_column is None:
    # Fall back to the first column if the file has only one logical
    # feature-name column.
    if feature_df.shape[1] >= 1:
        feature_column = feature_df.columns[0]
    else:
        raise ValueError("Could not identify the feature-name column.")

FINAL_FEATURES = feature_df[feature_column].astype(str).tolist()

print(f"Number of final features: {len(FINAL_FEATURES)}")

if len(FINAL_FEATURES) != 69:
    raise ValueError(
        f"Expected exactly 69 final model features, "
        f"found {len(FINAL_FEATURES)}."
    )

if len(set(FINAL_FEATURES)) != 69:
    raise ValueError("Duplicate features found in final feature list.")

print("✓ Exactly 69 final model features confirmed.")


# ================================================================
# LOAD TEST DATA
# ================================================================

print("\n" + "=" * 70)
print("LOADING FINAL TEST DATA FOR VISUALIZATION VERIFICATION")
print("=" * 70)

test_df = pd.read_csv(TEST_DATA_PATH)

print(f"Test dataset shape: {test_df.shape}")
print(f"Test observations: {len(test_df):,}")
print(f"Test columns: {len(test_df.columns)}")


# ================================================================
# TEST COLUMN CHECK
# ================================================================

print("\n" + "=" * 70)
print("SPATIAL-TEMPORAL COLUMN CHECK")
print("=" * 70)

required_spatial_temporal_columns = [
    "date",
    "latitude",
    "longitude",
    "chlorophyll_a"
]

missing_spatial_temporal = [
    col for col in required_spatial_temporal_columns
    if col not in test_df.columns
]

print(f"Missing spatial-temporal columns: {len(missing_spatial_temporal)}")

if missing_spatial_temporal:
    print("Missing columns:")
    for col in missing_spatial_temporal:
        print(f"  - {col}")

    raise ValueError(
        "Required spatial-temporal columns are missing."
    )

print("✓ Date, latitude, longitude and actual CHL are available.")


# ================================================================
# DATE VERIFICATION
# ================================================================

print("\n" + "=" * 70)
print("DATE / TIME VERIFICATION")
print("=" * 70)

test_df["date"] = pd.to_datetime(
    test_df["date"],
    errors="coerce"
)

date_nan = test_df["date"].isna().sum()

print(f"Invalid date values: {date_nan}")

if date_nan > 0:
    raise ValueError(
        "Invalid or missing date values detected."
    )

print(f"Minimum date: {test_df['date'].min()}")
print(f"Maximum date: {test_df['date'].max()}")
print(f"Unique dates: {test_df['date'].nunique():,}")

print("✓ Date values verified.")


# ================================================================
# SPATIAL VERIFICATION
# ================================================================

print("\n" + "=" * 70)
print("SPATIAL DATA VERIFICATION")
print("=" * 70)

latitude_nan = test_df["latitude"].isna().sum()
longitude_nan = test_df["longitude"].isna().sum()

latitude_inf = np.isinf(test_df["latitude"]).sum()
longitude_inf = np.isinf(test_df["longitude"]).sum()

print(f"Latitude NaN       : {latitude_nan}")
print(f"Longitude NaN      : {longitude_nan}")
print(f"Latitude infinite  : {latitude_inf}")
print(f"Longitude infinite : {longitude_inf}")

if (
    latitude_nan > 0
    or longitude_nan > 0
    or latitude_inf > 0
    or longitude_inf > 0
):
    raise ValueError(
        "Invalid spatial coordinates detected."
    )

print(f"Latitude range : {test_df['latitude'].min()} "
      f"to {test_df['latitude'].max()}")

print(f"Longitude range: {test_df['longitude'].min()} "
      f"to {test_df['longitude'].max()}")

print(f"Unique latitude locations : {test_df['latitude'].nunique():,}")
print(f"Unique longitude locations: {test_df['longitude'].nunique():,}")

print("✓ Spatial coordinates verified.")


# ================================================================
# ACTUAL CHL VERIFICATION
# ================================================================

print("\n" + "=" * 70)
print("ACTUAL CHL VERIFICATION")
print("=" * 70)

chl_nan = test_df["chlorophyll_a"].isna().sum()
chl_inf = np.isinf(test_df["chlorophyll_a"]).sum()

print(f"CHL NaN      : {chl_nan}")
print(f"CHL infinite : {chl_inf}")

if chl_nan > 0 or chl_inf > 0:
    raise ValueError(
        "Invalid actual CHL values detected."
    )

print(f"Minimum CHL: {test_df['chlorophyll_a'].min():.6f} mg/m³")
print(f"Maximum CHL: {test_df['chlorophyll_a'].max():.6f} mg/m³")
print(f"Mean CHL   : {test_df['chlorophyll_a'].mean():.6f} mg/m³")

print("✓ Actual CHL values verified.")


# ================================================================
# LOAD SAVED TUNED TEST PREDICTIONS
# ================================================================

print("\n" + "=" * 70)
print("LOADING SAVED TUNED XGBOOST TEST PREDICTIONS")
print("=" * 70)

prediction_df = pd.read_csv(
    TUNED_TEST_PREDICTIONS_PATH
)

print(f"Prediction dataset shape: {prediction_df.shape}")
print(f"Prediction observations: {len(prediction_df):,}")

print("\nPrediction columns:")
print(prediction_df.columns.tolist())


# ================================================================
# PREDICTION COLUMN CHECK
# ================================================================

print("\n" + "=" * 70)
print("PREDICTION COLUMN CHECK")
print("=" * 70)

required_prediction_columns = [
    "date",
    "latitude",
    "longitude",
    "actual_chlorophyll_a",
    "predicted_chlorophyll_a"
]

missing_prediction_columns = [
    col for col in required_prediction_columns
    if col not in prediction_df.columns
]

print(
    f"Missing required prediction columns: "
    f"{len(missing_prediction_columns)}"
)

if missing_prediction_columns:
    print("Missing:")
    for col in missing_prediction_columns:
        print(f"  - {col}")

    raise ValueError(
        "Required prediction columns are missing."
    )

print("✓ Required spatial-temporal prediction columns found.")


# ================================================================
# PREDICTION DATE VERIFICATION
# ================================================================

prediction_df["date"] = pd.to_datetime(
    prediction_df["date"],
    errors="coerce"
)

if prediction_df["date"].isna().sum() > 0:
    raise ValueError(
        "Invalid prediction dates detected."
    )

print(
    f"Prediction date range: "
    f"{prediction_df['date'].min()} "
    f"to "
    f"{prediction_df['date'].max()}"
)


# ================================================================
# PREDICTION SAFETY CHECK
# ================================================================

print("\n" + "=" * 70)
print("PREDICTION SAFETY CHECK")
print("=" * 70)

predicted_chl_nan = prediction_df[
    "predicted_chlorophyll_a"
].isna().sum()

predicted_chl_inf = np.isinf(
    prediction_df["predicted_chlorophyll_a"]
).sum()

actual_pred_chl_nan = prediction_df[
    "actual_chlorophyll_a"
].isna().sum()

print(f"Predicted CHL NaN      : {predicted_chl_nan}")
print(f"Predicted CHL infinite : {predicted_chl_inf}")
print(f"Actual CHL NaN         : {actual_pred_chl_nan}")

if (
    predicted_chl_nan > 0
    or predicted_chl_inf > 0
    or actual_pred_chl_nan > 0
):
    raise ValueError(
        "Invalid saved prediction values detected."
    )

print("✓ Saved tuned predictions passed safety checks.")


# ================================================================
# ROW COUNT CONSISTENCY CHECK
# ================================================================

print("\n" + "=" * 70)
print("ROW COUNT CONSISTENCY CHECK")
print("=" * 70)

print(f"Test dataset rows       : {len(test_df):,}")
print(f"Prediction dataset rows : {len(prediction_df):,}")

if len(test_df) != len(prediction_df):
    print(
        "⚠ Test and prediction row counts differ. "
        "This will require explicit spatial-temporal alignment."
    )
else:
    print("✓ Test and prediction row counts match.")


# ================================================================
# SPATIAL-TEMPORAL ALIGNMENT CHECK
# ================================================================

print("\n" + "=" * 70)
print("SPATIAL-TEMPORAL ALIGNMENT CHECK")
print("=" * 70)

test_keys = test_df[
    ["date", "latitude", "longitude"]
].copy()

prediction_keys = prediction_df[
    ["date", "latitude", "longitude"]
].copy()

test_keys = test_keys.sort_values(
    ["date", "latitude", "longitude"]
).reset_index(drop=True)

prediction_keys = prediction_keys.sort_values(
    ["date", "latitude", "longitude"]
).reset_index(drop=True)

if test_keys.equals(prediction_keys):
    print("✓ Date + latitude + longitude alignment confirmed.")
else:
    print(
        "⚠ Direct row-wise spatial-temporal alignment "
        "was not confirmed."
    )
    print(
        "The later visualization step will use an explicit "
        "key-based merge rather than assuming row order."
    )


# ================================================================
# VISUALIZATION DATA SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("3D VISUALIZATION DATA SUMMARY")
print("=" * 70)

print(f"Observations available : {len(prediction_df):,}")
print(
    f"Date range             : "
    f"{prediction_df['date'].min().date()} "
    f"→ "
    f"{prediction_df['date'].max().date()}"
)

print(
    f"Latitude range         : "
    f"{prediction_df['latitude'].min()} "
    f"→ "
    f"{prediction_df['latitude'].max()}"
)

print(
    f"Longitude range        : "
    f"{prediction_df['longitude'].min()} "
    f"→ "
    f"{prediction_df['longitude'].max()}"
)

print(
    f"Predicted CHL range    : "
    f"{prediction_df['predicted_chlorophyll_a'].min():.6f} "
    f"→ "
    f"{prediction_df['predicted_chlorophyll_a'].max():.6f} mg/m³"
)

print(
    f"Actual CHL range       : "
    f"{prediction_df['actual_chlorophyll_a'].min():.6f} "
    f"→ "
    f"{prediction_df['actual_chlorophyll_a'].max():.6f} mg/m³"
)


# ================================================================
# FUTURE VISUALIZATION ARCHITECTURE CHECK
# ================================================================

print("\n" + "=" * 70)
print("FUTURE 3D VISUALIZATION ARCHITECTURE")
print("=" * 70)

print("""
Historical CHL
      ↓
Current/API CHL
      ↓
Day +1 predicted CHL
      ↓
Day +2 predicted CHL
      ↓
Day +3 predicted CHL
      ↓
Interactive time dimension
      ↓
Latitude × Longitude × CHL
      ↓
Spatial-temporal CHL change visualization
""")

print("✓ Current dataset structure can support the historical/current")
print("  visualization stage.")
print("✓ Future API predictions will be integrated in later steps.")
print("✓ No future predictions are generated in this step.")


# ================================================================
# SAFETY CHECK
# ================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16T-17 SAFETY CHECK")
print("=" * 70)

print("Final XGBoost retrained       : FALSE")
print("Hyperparameters changed       : FALSE")
print("Final model modified          : FALSE")
print("Test predictions regenerated  : FALSE")
print("SHAP recalculated             : FALSE")
print("API data requested            : FALSE")
print("Future CHL predicted          : FALSE")
print("Historical/test data used     : TRUE")
print("Purpose                       : 3D visualization verification")

print("\n✓ Final XGBoost remains untouched.")
print("✓ Existing tuned predictions were only read.")
print("✓ Spatial-temporal data verified.")
print("✓ Visualization environment verified.")
print("✓ Ready for 3D visualization construction.")

print("\n" + "=" * 70)
print("STEP 31.9P-16T-17 COMPLETE")
print("=" * 70)

print("\nNEXT:")
print("31.9P-16T-18 — Historical/Current CHL 3D Visualization")

STEP 31.9P-16T-17
3D SPATIAL-TEMPORAL CHL VISUALIZATION
ENVIRONMENT & DATA VERIFICATION

PROJECT PATHS

Project:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Final tuned model:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate/xgboost_final_tuned_candidate.json

Final feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv

Final test dataset:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_test.csv

Saved tuned test predictions:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate/final_test_evaluation/xgboost_final_tuned_test_predictions.csv

SHAP directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate/shap_explainability

3D visualization output directory:
/cont

In [11]:
# ======================================================================
# STEP 31.9P-16T-18
# INTERACTIVE 3D SPATIAL CHL SURFACE
# FINAL REFERENCE-STYLE VISUALIZATION
# ======================================================================
#
# IMPORTANT METHODOLOGICAL SAFETY:
# - Existing final tuned XGBoost predictions are reused
# - XGBoost is NOT retrained
# - XGBoost parameters are NOT changed
# - Original CHL values are NOT changed
# - Interpolation is visualization-only
# - Vertical exaggeration is visualization-only
# - Actual and predicted CHL use the SAME colour scale
# - Actual and predicted surfaces use the SAME vertical scale
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from scipy.interpolate import RBFInterpolator
from scipy.spatial import Delaunay
from scipy.ndimage import gaussian_filter


# ======================================================================
# 1. PROJECT PATHS
# ======================================================================

PROJECT_ROOT = (
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

PREDICTION_FILE = (
    f"{PROJECT_ROOT}/06_models/02_XGBoost/"
    f"hyperparameter_optimization/final_tuned_candidate/"
    f"final_test_evaluation/"
    f"xgboost_final_tuned_test_predictions.csv"
)

OUTPUT_DIR = (
    f"{PROJECT_ROOT}/06_models/02_XGBoost/"
    f"hyperparameter_optimization/final_tuned_candidate/"
    f"3d_spatial_temporal_visualization"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ======================================================================
# 2. STEP HEADER
# ======================================================================

print("=" * 75)
print("STEP 31.9P-16T-18")
print("FINAL INTERACTIVE 3D SPATIAL CHL VISUALIZATION")
print("=" * 75)

print("\nVisualization design:")
print("  ✓ RBF spatial interpolation")
print("  ✓ Convex-hull geographic masking")
print("  ✓ Common CHL colour scale")
print("  ✓ Common vertical CHL scale")
print("  ✓ Smooth scientific surface")
print("  ✓ Original observation points")
print("  ✓ Interactive Plotly visualization")


# ======================================================================
# 3. LOAD EXISTING FINAL TUNED XGBOOST PREDICTIONS
# ======================================================================

print("\n" + "=" * 75)
print("LOADING EXISTING TUNED XGBOOST PREDICTIONS")
print("=" * 75)

if not os.path.exists(PREDICTION_FILE):
    raise FileNotFoundError(
        f"Prediction file not found:\n{PREDICTION_FILE}"
    )

df = pd.read_csv(PREDICTION_FILE)

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

required_columns = [
    "date",
    "latitude",
    "longitude",
    "actual_chlorophyll_a",
    "predicted_chlorophyll_a"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print(f"Dataset shape : {df.shape}")
print(
    f"Date range    : "
    f"{df['date'].min()} → {df['date'].max()}"
)


# ======================================================================
# 4. BASIC DATA SAFETY CHECK
# ======================================================================

print("\n" + "=" * 75)
print("DATA SAFETY CHECK")
print("=" * 75)

if df[required_columns].isna().any().any():
    raise ValueError(
        "Required visualization columns contain missing values."
    )

if np.isinf(
    df[
        [
            "latitude",
            "longitude",
            "actual_chlorophyll_a",
            "predicted_chlorophyll_a"
        ]
    ].to_numpy()
).any():
    raise ValueError(
        "Required visualization columns contain infinite values."
    )

print("✓ Required columns available.")
print("✓ No NaN values in required columns.")
print("✓ No infinite values in required columns.")


# ======================================================================
# 5. SELECT VISUALIZATION DATE
# ======================================================================
#
# For this stage we visualize the latest available historical/test date.
#
# Later, 31.9P-16T-19 will replace this fixed date with an interactive
# time dimension.
# ======================================================================

selected_date = df["date"].max()

day_df = (
    df[
        df["date"] == selected_date
    ]
    .copy()
    .dropna(
        subset=[
            "latitude",
            "longitude",
            "actual_chlorophyll_a",
            "predicted_chlorophyll_a"
        ]
    )
)

if len(day_df) < 4:
    raise ValueError(
        "Insufficient spatial observations for 3D interpolation."
    )

print("\n" + "=" * 75)
print("SELECTED SPATIAL-TEMPORAL SLICE")
print("=" * 75)

print(
    f"Selected date : "
    f"{selected_date.strftime('%Y-%m-%d')}"
)

print(f"Observations  : {len(day_df)}")
print(
    f"Latitudes     : "
    f"{day_df['latitude'].nunique()}"
)
print(
    f"Longitudes    : "
    f"{day_df['longitude'].nunique()}"
)


# ======================================================================
# 6. EXTRACT SPATIAL VARIABLES
# ======================================================================

lat = day_df["latitude"].to_numpy(dtype=float)
lon = day_df["longitude"].to_numpy(dtype=float)

actual = day_df[
    "actual_chlorophyll_a"
].to_numpy(dtype=float)

predicted = day_df[
    "predicted_chlorophyll_a"
].to_numpy(dtype=float)


# ======================================================================
# 7. COMMON CHL SCALE
# ======================================================================
#
# IMPORTANT:
# The SAME colour range is used for:
#
#   Actual CHL
#   Predicted CHL
#   Actual observation points
#   Predicted observation points
#
# This makes the two visualizations directly comparable.
# ======================================================================

CHL_MIN = min(
    np.nanmin(actual),
    np.nanmin(predicted)
)

CHL_MAX = max(
    np.nanmax(actual),
    np.nanmax(predicted)
)

if CHL_MAX <= CHL_MIN:
    raise ValueError(
        "Invalid common CHL range."
    )

print("\n" + "=" * 75)
print("COMMON CHL SCALE")
print("=" * 75)

print(
    f"Actual CHL range    : "
    f"{np.nanmin(actual):.6f} → "
    f"{np.nanmax(actual):.6f} mg/m³"
)

print(
    f"Predicted CHL range : "
    f"{np.nanmin(predicted):.6f} → "
    f"{np.nanmax(predicted):.6f} mg/m³"
)

print(
    f"COMMON colour scale : "
    f"{CHL_MIN:.6f} → "
    f"{CHL_MAX:.6f} mg/m³"
)

print(
    "✓ Actual and predicted use the same CHL colour scale."
)


# ======================================================================
# 8. HIGH-RESOLUTION SPATIAL GRID
# ======================================================================

GRID_SIZE = 180

grid_lon = np.linspace(
    lon.min(),
    lon.max(),
    GRID_SIZE
)

grid_lat = np.linspace(
    lat.min(),
    lat.max(),
    GRID_SIZE
)

GRID_LON, GRID_LAT = np.meshgrid(
    grid_lon,
    grid_lat
)

print("\n" + "=" * 75)
print("SPATIAL GRID")
print("=" * 75)

print(
    f"Grid size : "
    f"{GRID_SIZE} × {GRID_SIZE}"
)


# ======================================================================
# 9. PREPARE SPATIAL POINTS
# ======================================================================

points = np.column_stack(
    [
        lon,
        lat
    ]
)

grid_points = np.column_stack(
    [
        GRID_LON.ravel(),
        GRID_LAT.ravel()
    ]
)


# ======================================================================
# 10. GEOGRAPHIC CONVEX-HULL MASK
# ======================================================================
#
# This prevents the visualization from inventing CHL values outside
# the spatial region supported by the observations.
# ======================================================================

print("\n" + "=" * 75)
print("GEOGRAPHIC SUPPORT REGION")
print("=" * 75)

try:

    triangulation = Delaunay(points)

    inside_hull = (
        triangulation.find_simplex(
            grid_points
        ) >= 0
    )

except Exception as e:

    raise ValueError(
        f"Could not create spatial support region: {e}"
    )

inside_hull_2d = inside_hull.reshape(
    GRID_LON.shape
)

print(
    f"Supported grid cells : "
    f"{inside_hull.sum()} / "
    f"{len(inside_hull)}"
)

print(
    "✓ Convex-hull spatial masking enabled."
)


# ======================================================================
# 11. RBF INTERPOLATION — PREDICTED CHL
# ======================================================================
#
# RBF is used to create a smooth continuous surface from the existing
# spatial observations.
#
# This is ONLY a visualization operation.
# The saved XGBoost predictions are not changed.
# ======================================================================

print("\n" + "=" * 75)
print("RBF INTERPOLATION — PREDICTED CHL")
print("=" * 75)

rbf_predicted = RBFInterpolator(
    points,
    predicted,
    kernel="thin_plate_spline",
    smoothing=0.06,
    neighbors=min(25, len(points))
)

predicted_surface = (
    rbf_predicted(
        grid_points
    )
    .reshape(GRID_LON.shape)
)

print("✓ Predicted RBF surface created.")


# ======================================================================
# 12. RBF INTERPOLATION — ACTUAL CHL
# ======================================================================

print("\n" + "=" * 75)
print("RBF INTERPOLATION — ACTUAL CHL")
print("=" * 75)

rbf_actual = RBFInterpolator(
    points,
    actual,
    kernel="thin_plate_spline",
    smoothing=0.06,
    neighbors=min(25, len(points))
)

actual_surface = (
    rbf_actual(
        grid_points
    )
    .reshape(GRID_LON.shape)
)

print("✓ Actual RBF surface created.")


# ======================================================================
# 13. APPLY GEOGRAPHIC MASK
# ======================================================================

predicted_surface[
    ~inside_hull_2d
] = np.nan

actual_surface[
    ~inside_hull_2d
] = np.nan


# ======================================================================
# 14. LIGHT VISUAL SMOOTHING
# ======================================================================
#
# NaN cells are temporarily filled only for the smoothing operation.
# The geographic mask is restored immediately afterwards.
# ======================================================================

print("\n" + "=" * 75)
print("SURFACE SMOOTHING")
print("=" * 75)

predicted_fill_value = np.nanmean(
    predicted_surface
)

actual_fill_value = np.nanmean(
    actual_surface
)

predicted_filled = np.where(
    np.isnan(predicted_surface),
    predicted_fill_value,
    predicted_surface
)

actual_filled = np.where(
    np.isnan(actual_surface),
    actual_fill_value,
    actual_surface
)

predicted_smooth = gaussian_filter(
    predicted_filled,
    sigma=1.2
)

actual_smooth = gaussian_filter(
    actual_filled,
    sigma=1.2
)

# Restore unsupported geographic region
predicted_smooth[
    ~inside_hull_2d
] = np.nan

actual_smooth[
    ~inside_hull_2d
] = np.nan

print("✓ Light surface smoothing applied.")
print("✓ Geographic boundary restored.")


# ======================================================================
# 15. COMMON VERTICAL SCALE
# ======================================================================
#
# IMPORTANT:
# Actual and predicted surfaces are transformed using the SAME CHL
# range.
#
# Therefore:
#
#   Same CHL value → same visual height
#
# This makes Actual vs Predicted spatial comparison meaningful.
#
# The vertical exaggeration is visual only.
# ======================================================================

VERTICAL_EXAGGERATION = 4.0

predicted_z = (
    (predicted_smooth - CHL_MIN)
    / (CHL_MAX - CHL_MIN)
) * VERTICAL_EXAGGERATION

actual_z = (
    (actual_smooth - CHL_MIN)
    / (CHL_MAX - CHL_MIN)
) * VERTICAL_EXAGGERATION

print("\n" + "=" * 75)
print("COMMON VERTICAL SCALE")
print("=" * 75)

print(
    f"Vertical exaggeration : "
    f"{VERTICAL_EXAGGERATION}×"
)

print(
    f"Common Z range        : "
    f"0 → {VERTICAL_EXAGGERATION:.1f}"
)

print(
    "✓ Actual and predicted use the same vertical CHL transformation."
)


# ======================================================================
# 16. OBSERVATION-POINT Z VALUES
# ======================================================================
#
# The original spatial observations are placed on the SAME visual scale
# as the interpolated surface.
# ======================================================================

actual_z_points = (
    (actual - CHL_MIN)
    / (CHL_MAX - CHL_MIN)
) * VERTICAL_EXAGGERATION

predicted_z_points = (
    (predicted - CHL_MIN)
    / (CHL_MAX - CHL_MIN)
) * VERTICAL_EXAGGERATION


# ======================================================================
# 17. CUSTOM HOVER VALUES
# ======================================================================
#
# Hover displays REAL CHL values, not exaggerated visual height.
# ======================================================================

predicted_hover = np.round(
    predicted_smooth,
    4
)

actual_hover = np.round(
    actual_smooth,
    4
)


# ======================================================================
# 18. COMMON CAMERA
# ======================================================================
#
# The SAME camera is used for both Actual and Predicted plots.
# ======================================================================

COMMON_CAMERA = dict(
    eye=dict(
        x=1.8,
        y=1.8,
        z=1.35
    ),
    center=dict(
        x=0,
        y=0,
        z=0
    )
)


# ======================================================================
# 19. COMMON 3D SCENE
# ======================================================================

def create_common_scene():

    return dict(

        bgcolor="#172535",

        xaxis=dict(
            title="Longitude (°E)",
            backgroundcolor="#203449",
            gridcolor="rgba(255,255,255,0.25)",
            zerolinecolor="rgba(255,255,255,0.35)"
        ),

        yaxis=dict(
            title="Latitude (°N)",
            backgroundcolor="#203449",
            gridcolor="rgba(255,255,255,0.25)",
            zerolinecolor="rgba(255,255,255,0.35)"
        ),

        zaxis=dict(
            title="CHL (mg/m³) — vertically exaggerated",
            range=[
                0,
                VERTICAL_EXAGGERATION
            ],
            backgroundcolor="#203449",
            gridcolor="rgba(255,255,255,0.25)",
            zerolinecolor="rgba(255,255,255,0.35)"
        ),

        aspectmode="manual",

        aspectratio=dict(
            x=1.45,
            y=1.0,
            z=0.65
        ),

        camera=COMMON_CAMERA
    )


# ======================================================================
# 20. PREDICTED CHL FIGURE
# ======================================================================

print("\n" + "=" * 75)
print("CREATING PREDICTED CHL 3D VISUALIZATION")
print("=" * 75)

fig_predicted = go.Figure()


# ----------------------------------------------------------------------
# Predicted smooth surface
# ----------------------------------------------------------------------

fig_predicted.add_trace(

    go.Surface(

        x=GRID_LON,
        y=GRID_LAT,
        z=predicted_z,

        surfacecolor=predicted_smooth,

        colorscale="Turbo",

        cmin=CHL_MIN,
        cmax=CHL_MAX,

        colorbar=dict(
            title=dict(
                text="CHL<br>(mg/m³)"
            ),
            thickness=18,
            len=0.72
        ),

        customdata=predicted_hover,

        hovertemplate=(
            "<b>Predicted CHL Surface</b><br>"
            "Longitude: %{x:.3f}°E<br>"
            "Latitude: %{y:.3f}°N<br>"
            "CHL: %{customdata:.4f} mg/m³"
            "<extra></extra>"
        ),

        lighting=dict(
            ambient=0.50,
            diffuse=0.75,
            specular=0.30,
            roughness=0.55,
            fresnel=0.10
        ),

        contours=dict(
            x=dict(show=False),
            y=dict(show=False),
            z=dict(show=False)
        ),

        name="Predicted CHL"
    )
)


# ----------------------------------------------------------------------
# Predicted observation locations
# ----------------------------------------------------------------------

fig_predicted.add_trace(

    go.Scatter3d(

        x=lon,
        y=lat,
        z=predicted_z_points,

        mode="markers",

        marker=dict(
            size=3.5,
            color=predicted,
            colorscale="Turbo",
            cmin=CHL_MIN,
            cmax=CHL_MAX,
            opacity=0.80
        ),

        customdata=np.column_stack(
            [
                predicted,
                lat,
                lon
            ]
        ),

        hovertemplate=(
            "<b>Prediction Location</b><br>"
            "Longitude: %{customdata[2]:.3f}°E<br>"
            "Latitude: %{customdata[1]:.3f}°N<br>"
            "Predicted CHL: "
            "%{customdata[0]:.4f} mg/m³"
            "<extra></extra>"
        ),

        name="Prediction Locations"
    )
)


# ----------------------------------------------------------------------
# Predicted layout
# ----------------------------------------------------------------------

fig_predicted.update_layout(

    title=dict(

        text=(
            "<b>Interactive 3D Spatial CHL Distribution</b><br>"
            f"<sup>Predicted CHL — "
            f"{selected_date.strftime('%Y-%m-%d')}</sup>"
        ),

        x=0.03,
        xanchor="left",

        y=0.96,
        yanchor="top"
    ),

    paper_bgcolor="#172535",
    plot_bgcolor="#172535",

    font=dict(
        color="white",
        size=12
    ),

    scene=create_common_scene(),

    margin=dict(
        l=0,
        r=0,
        t=90,
        b=0
    ),

    legend=dict(
        bgcolor="rgba(0,0,0,0.35)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1
    )
)


# ======================================================================
# 21. SAVE PREDICTED VISUALIZATION
# ======================================================================

predicted_output = os.path.join(
    OUTPUT_DIR,
    "final_predicted_chl_3d.html"
)

fig_predicted.write_html(
    predicted_output,
    include_plotlyjs=True
)

print(
    "\n✓ Predicted 3D visualization saved:"
)

print(predicted_output)

fig_predicted.show()


# ======================================================================
# 22. ACTUAL CHL FIGURE
# ======================================================================

print("\n" + "=" * 75)
print("CREATING ACTUAL CHL 3D VISUALIZATION")
print("=" * 75)

fig_actual = go.Figure()


# ----------------------------------------------------------------------
# Actual smooth surface
# ----------------------------------------------------------------------

fig_actual.add_trace(

    go.Surface(

        x=GRID_LON,
        y=GRID_LAT,
        z=actual_z,

        surfacecolor=actual_smooth,

        colorscale="Turbo",

        # IMPORTANT:
        # SAME CHL RANGE AS PREDICTED
        cmin=CHL_MIN,
        cmax=CHL_MAX,

        colorbar=dict(
            title=dict(
                text="CHL<br>(mg/m³)"
            ),
            thickness=18,
            len=0.72
        ),

        customdata=actual_hover,

        hovertemplate=(
            "<b>Actual CHL Surface</b><br>"
            "Longitude: %{x:.3f}°E<br>"
            "Latitude: %{y:.3f}°N<br>"
            "CHL: %{customdata:.4f} mg/m³"
            "<extra></extra>"
        ),

        lighting=dict(
            ambient=0.50,
            diffuse=0.75,
            specular=0.30,
            roughness=0.55,
            fresnel=0.10
        ),

        contours=dict(
            x=dict(show=False),
            y=dict(show=False),
            z=dict(show=False)
        ),

        name="Actual CHL"
    )
)


# ----------------------------------------------------------------------
# Actual observation locations
# ----------------------------------------------------------------------

fig_actual.add_trace(

    go.Scatter3d(

        x=lon,
        y=lat,
        z=actual_z_points,

        mode="markers",

        marker=dict(
            size=3.5,
            color=actual,

            # IMPORTANT:
            # SAME COLOUR SCALE AS PREDICTED
            colorscale="Turbo",
            cmin=CHL_MIN,
            cmax=CHL_MAX,

            opacity=0.80
        ),

        customdata=np.column_stack(
            [
                actual,
                lat,
                lon
            ]
        ),

        hovertemplate=(
            "<b>Observation Location</b><br>"
            "Longitude: %{customdata[2]:.3f}°E<br>"
            "Latitude: %{customdata[1]:.3f}°N<br>"
            "Actual CHL: "
            "%{customdata[0]:.4f} mg/m³"
            "<extra></extra>"
        ),

        name="Observation Locations"
    )
)


# ----------------------------------------------------------------------
# Actual layout
# ----------------------------------------------------------------------

fig_actual.update_layout(

    title=dict(

        text=(
            "<b>Interactive 3D Spatial CHL Distribution</b><br>"
            f"<sup>Actual CHL — "
            f"{selected_date.strftime('%Y-%m-%d')}</sup>"
        ),

        x=0.03,
        xanchor="left",

        y=0.96,
        yanchor="top"
    ),

    paper_bgcolor="#172535",
    plot_bgcolor="#172535",

    font=dict(
        color="white",
        size=12
    ),

    scene=create_common_scene(),

    margin=dict(
        l=0,
        r=0,
        t=90,
        b=0
    ),

    legend=dict(
        bgcolor="rgba(0,0,0,0.35)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1
    )
)


# ======================================================================
# 23. SAVE ACTUAL VISUALIZATION
# ======================================================================

actual_output = os.path.join(
    OUTPUT_DIR,
    "final_actual_chl_3d.html"
)

fig_actual.write_html(
    actual_output,
    include_plotlyjs=True
)

print(
    "\n✓ Actual 3D visualization saved:"
)

print(actual_output)

fig_actual.show()


# ======================================================================
# 24. SAVE VISUALIZATION SURFACE DATA
# ======================================================================

surface_data = pd.DataFrame({

    "latitude": GRID_LAT.ravel(),

    "longitude": GRID_LON.ravel(),

    "predicted_chl": predicted_smooth.ravel(),

    "actual_chl": actual_smooth.ravel(),

    "predicted_visual_height": predicted_z.ravel(),

    "actual_visual_height": actual_z.ravel(),

    "inside_spatial_support": inside_hull.ravel()

})

surface_data_output = os.path.join(
    OUTPUT_DIR,
    "final_3d_chl_surface_data.csv"
)

surface_data.to_csv(
    surface_data_output,
    index=False
)

print(
    "\n✓ 3D surface data saved:"
)

print(surface_data_output)


# ======================================================================
# 25. SAVE VISUALIZATION METADATA
# ======================================================================

metadata = {

    "step": "31.9P-16T-18",

    "selected_date":
        selected_date.strftime("%Y-%m-%d"),

    "observations":
        int(len(day_df)),

    "grid_size":
        int(GRID_SIZE),

    "interpolation":
        "RBF thin_plate_spline",

    "rbf_smoothing":
        0.06,

    "gaussian_smoothing_sigma":
        1.2,

    "common_chl_min":
        float(CHL_MIN),

    "common_chl_max":
        float(CHL_MAX),

    "vertical_exaggeration":
        float(VERTICAL_EXAGGERATION),

    "actual_chl_min":
        float(np.nanmin(actual)),

    "actual_chl_max":
        float(np.nanmax(actual)),

    "predicted_chl_min":
        float(np.nanmin(predicted)),

    "predicted_chl_max":
        float(np.nanmax(predicted)),

    "model_retrained":
        False,

    "predictions_regenerated":
        False,

    "model_modified":
        False,

    "api_data_requested":
        False,

    "purpose":
        "Historical/current spatial CHL visualization"
}

metadata_output = os.path.join(
    OUTPUT_DIR,
    "final_3d_chl_visualization_metadata.json"
)

import json

with open(
    metadata_output,
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

print(
    "\n✓ Visualization metadata saved:"
)

print(metadata_output)


# ======================================================================
# 26. FINAL SAFETY CHECK
# ======================================================================

print("\n" + "=" * 75)
print("FINAL 31.9P-16T-18 SAFETY CHECK")
print("=" * 75)

print(
    "Existing tuned predictions reused       : TRUE"
)

print(
    "XGBoost retrained                       : FALSE"
)

print(
    "XGBoost parameters changed              : FALSE"
)

print(
    "New model created                       : FALSE"
)

print(
    "Predictions regenerated                 : FALSE"
)

print(
    "SHAP recalculated                       : FALSE"
)

print(
    "API data requested                      : FALSE"
)

print(
    "Actual CHL values preserved             : TRUE"
)

print(
    "Predicted CHL values preserved          : TRUE"
)

print(
    "RBF spatial interpolation               : TRUE"
)

print(
    "Convex-hull spatial masking             : TRUE"
)

print(
    "Visualization smoothing                 : TRUE"
)

print(
    "Common CHL colour scale                 : TRUE"
)

print(
    "Common vertical scale                   : TRUE"
)

print(
    "Vertical visual exaggeration            : TRUE"
)

print(
    "Interactive 3D surfaces                 : TRUE"
)

print(
    "Original spatial observations displayed : TRUE"
)

print(
    "\n✓ Actual and predicted plots use the SAME CHL scale."
)

print(
    "✓ Actual and predicted plots use the SAME Z scale."
)

print(
    "✓ No artificial nearest-neighbour extension is used."
)

print(
    "✓ Final XGBoost remains completely untouched."
)

print(
    "\n✓ 31.9P-16T-18 COMPLETE"
)

print("=" * 75)

STEP 31.9P-16T-18
FINAL INTERACTIVE 3D SPATIAL CHL VISUALIZATION

Visualization design:
  ✓ RBF spatial interpolation
  ✓ Convex-hull geographic masking
  ✓ Common CHL colour scale
  ✓ Common vertical CHL scale
  ✓ Smooth scientific surface
  ✓ Original observation points
  ✓ Interactive Plotly visualization

LOADING EXISTING TUNED XGBOOST PREDICTIONS
Dataset shape : (157998, 7)
Date range    : 2021-10-05 00:00:00 → 2025-12-31 00:00:00

DATA SAFETY CHECK
✓ Required columns available.
✓ No NaN values in required columns.
✓ No infinite values in required columns.

SELECTED SPATIAL-TEMPORAL SLICE
Selected date : 2025-12-31
Observations  : 102
Latitudes     : 14
Longitudes    : 9

COMMON CHL SCALE
Actual CHL range    : 0.160934 → 0.299414 mg/m³
Predicted CHL range : 0.154710 → 0.353699 mg/m³
COMMON colour scale : 0.154710 → 0.353699 mg/m³
✓ Actual and predicted use the same CHL colour scale.

SPATIAL GRID
Grid size : 180 × 180

GEOGRAPHIC SUPPORT REGION
Supported grid cells : 28613 / 32400


CREATING ACTUAL CHL 3D VISUALIZATION

✓ Actual 3D visualization saved:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate/3d_spatial_temporal_visualization/final_actual_chl_3d.html



✓ 3D surface data saved:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate/3d_spatial_temporal_visualization/final_3d_chl_surface_data.csv

✓ Visualization metadata saved:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate/3d_spatial_temporal_visualization/final_3d_chl_visualization_metadata.json

FINAL 31.9P-16T-18 SAFETY CHECK
Existing tuned predictions reused       : TRUE
XGBoost retrained                       : FALSE
XGBoost parameters changed              : FALSE
New model created                       : FALSE
Predictions regenerated                 : FALSE
SHAP recalculated                       : FALSE
API data requested                      : FALSE
Actual CHL values preserved             : TRUE
Predicted CHL values preserved          : TRUE
RBF spatial interpolation               : TRUE
Convex-hull spatial masking             : TRUE
Visu